# exp_013 — FRONTIER sweep (Colab, direct zip download)

Targets the **unsolved cells**: the L2/L3 levels where BOTH standard baselines (ICM and RND)
scored **0/8** in the calibration sweep. Random policy E = ∞ on all of these. A single solve
here is the whole result.

| cell | random E | icm | rnd |
|---|---|---|---|
| ls20 L2 | ∞ | 0/8 | 0/8 |
| ls20 L3 | ∞ | 0/8 | 0/8 |
| g50t L2 | ∞ | 0/8 | 0/8 |
| g50t L3 | ∞ | 0/8 | 0/8 |
| re86 L2 | ∞ | 0/8 | 0/8 |
| re86 L3 | ∞ | 0/8 | 0/8 |

**Methods (the new ideas):**
| tag | what | why on the frontier |
|---|---|---|
| **A** `frozen-φ` | RND+leak on a **frozen-random** φ (no ICM) | needs no φ-controllability — sidesteps the broken-ICM-φ failure on ls20; most stable in diagnosis |
| **B** `rnd_icm` | RND+leak on **ICM** φ | the main method |
| **C** `additive` | `½·ICM + ½·RND-on-φ` | solved g50t L1 2/2 fast; cheap control |
| **D** `lookahead` | actor-free 1-step lookahead softmax | structurally **cannot entropy-collapse** (no policy gradient) |

**Fixes folded in since the L1 runs (from `probes/run_diagnosis.md`):** D's τ 1.0→**0.25** (τ=1.0 was a near-uniform random walk); B/C's φ-freeze gate 0.90→**0.70** (0.90 was unreachable → froze on a chance-level φ); c_entropy 0.05→**0.10** (long runs collapsed onto the frozen-φ degenerate signal). E (disagreement) excluded — never validated beyond a smoke.

> **Set Runtime ▸ GPU.** Downloads the results zip **directly to your browser** (no Google Drive).

## 1. Setup (clone FIRST, then install)

In [ ]:
import os, sys, glob, json, time
REPO_URL = "https://github.com/LavetteSinsora/ProjectArceus.git"; REPO = "/content/ProjectArceus"
if not os.path.isdir(REPO):
    !git clone --depth 1 $REPO_URL $REPO
%cd /content/ProjectArceus
!git pull --ff-only -q || true
!pip -q install "arc-agi>=0.9.8" "arcengine>=0.9.3"
!pip -q install -e . --no-deps
import torch; print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(set Runtime→GPU!)")

## 2. Sanity pre-flight — tu93 L2 (cheap)

Before spending the censored-run budget on the hard L2/L3, confirm the methods actually work on
**tu93 L2** — a cell the baselines solve in ~2k steps. A/B/C/D should solve it fast; if a method
censors here it's broken, don't trust its frontier result. Set `RUN_SANITY=False` to skip.

In [ ]:
RUN_SANITY = True
if RUN_SANITY:
    sanity = ["--games", "tu93", "--levels", "1", "--methods", "A", "B", "C", "D",
              "--seeds", "0", "--cap", "40000", "--no-transfer", "--concurrency", "2",
              "--logdir", "/content/exp013_sanity_logs"]
    SANITY = " ".join(sanity)
    print("sanity:", SANITY)
    !python -m JEPA.experiments.exp_013_headline_experiment.sweep $SANITY
else:
    print("skipped sanity")

## 3. Frontier config (edit for your budget)

Censored runs go to the full `CAP`, so cost ≈ `len(GAMES)×len(LEVELS)×len(METHODS)×len(SEEDS) × CAP-time / CONCURRENCY`.
Trim by dropping a game, using `SEEDS=[0]`, or lowering `CAP`. `--no-transfer` keeps it to just the frontier cells.

In [ ]:
GAMES   = ["ls20", "g50t", "re86"]   # frontier games
LEVELS  = [1, 2]                       # 0-indexed -> L2, L3
METHODS = ["A", "B", "C", "D"]        # add "E" (disagreement) only if you have spare budget
SEEDS   = [0, 1]
CAP     = 400_000                      # uniform per-cell step cap (override the per-cell CAPS dict)
CONCURRENCY = 2                        # these models are small; 2-3 parallel runs use a GPU's headroom
N_ENVS  = None                         # None = method default (16)

args = ["--games", *GAMES, "--levels", *map(str, LEVELS), "--methods", *METHODS,
        "--seeds", *map(str, SEEDS), "--cap", str(CAP), "--no-transfer",
        "--concurrency", str(CONCURRENCY), "--logdir", "/content/exp013_frontier_logs"]
if N_ENVS: args += ["--n-envs", str(N_ENVS)]
ARGS = " ".join(args)
n_runs = len(GAMES) * len(LEVELS) * len(METHODS) * len(SEEDS)
print("sweep args:", ARGS)
print(f"{n_runs} runs total | cap {CAP:,} | concurrency {CONCURRENCY}")

## 4. Run the frontier sweep

Re-runnable (each run writes its own timestamped dir). `sweep.py` prints a per-run log line and a
final solve-rate table vs the random benchmark. Output streams live below.

In [ ]:
!python -m JEPA.experiments.exp_013_headline_experiment.sweep $ARGS

## 5. Download the results zip (direct to browser)

In [ ]:
import shutil
stamp = time.strftime("%Y%m%d_%H%M%S")
zip_path = f"/content/exp013_frontier_{stamp}.zip"
shutil.make_archive(zip_path[:-4], "zip", "JEPA/experiments/exp_013_headline_experiment")
print("zipped ->", zip_path, f"({os.path.getsize(zip_path)/1e6:.1f} MB)")
try:
    from google.colab import files; files.download(zip_path)   # direct browser download
except Exception as e:
    print("Auto-download failed (run this cell with the tab focused, or use the Files pane):", zip_path, e)